<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap07/7_2_Backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 7.2: Backpropagation**

This notebook runs the backpropagation algorithm on a deep neural network as described in section 7.4 of the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

First let's define a neural network.  We'll just choose the weights and biases randomly for now

In [2]:
# Set seed so we always get the same random numbers
np.random.seed(0)

# Number of hidden layers
K = 5
# Number of neurons per layer
D = 6
# Input layer
D_i = 1
# Output layer
D_o = 1

# Make empty lists
all_weights = [None] * (K+1)
all_biases = [None] * (K+1)

# Create input and output layers
all_weights[0] = np.random.normal(size=(D, D_i))
all_weights[-1] = np.random.normal(size=(D_o, D))
all_biases[0] = np.random.normal(size =(D,1))
all_biases[-1]= np.random.normal(size =(D_o,1))

# Create intermediate layers
for layer in range(1,K):
  all_weights[layer] = np.random.normal(size=(D,D))
  all_biases[layer] = np.random.normal(size=(D,1))

In [3]:
# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation

Now let's run our random network.  The weight matrices $\boldsymbol\Omega_{0\ldots K}$ are the entries of the list "all_weights" and the biases $\boldsymbol\beta_{0\ldots K}$ are the entries of the list "all_biases"

We know that we will need the preactivations $\mathbf{f}_{0\ldots K}$ and the activations $\mathbf{h}_{1\ldots K}$ for the forward pass of backpropagation, so we'll store and return these as well.


In [4]:
def compute_network_output(net_input, all_weights, all_biases):

  # Retrieve number of layers
  K = len(all_weights) -1

  # We'll store the pre-activations at each layer in a list "all_f"
  # and the activations in a second list "all_h".
  all_f = [None] * (K+1)
  all_h = [None] * (K+1)

  #For convenience, we'll set
  # all_h[0] to be the input, and all_f[K] will be the output
  all_h[0] = net_input

  # Run through the layers, calculating all_f[0...K-1] and all_h[1...K]
  for layer in range(K):
      # Update preactivations and activations at this layer according to eqn 7.17
      # Remember to use np.matmul for matrix multiplications
      # TODO -- Replace the lines below
      all_f[layer] = all_biases[layer] + np.matmul(all_weights[layer], all_h[layer])
      all_h[layer+1] = ReLU(all_f[layer])

  # Compute the output from the last hidden layer
  # TODO -- Replace the line below
  all_f[K] = all_biases[K] + np.matmul(all_weights[K], all_h[K])

  # Retrieve the output
  net_output = all_f[K]

  return net_output, all_f, all_h

In [5]:
# Define input
net_input = np.ones((D_i,1)) * 1.2
# Compute network output
net_output, all_f, all_h = compute_network_output(net_input,all_weights, all_biases)
print("True output = %3.3f, Your answer = %3.3f"%(1.907, net_output[0,0]))

True output = 1.907, Your answer = 1.907


Now let's define a loss function.  We'll just use the least squares loss function. We'll also write a function to compute dloss_doutput

In [6]:
def least_squares_loss(net_output, y):
  return np.sum((net_output-y) * (net_output-y))

def d_loss_d_output(net_output, y):
    return 2*(net_output -y);

In [7]:
y = np.ones((D_o,1)) * 20.0
loss = least_squares_loss(net_output, y)
print("y = %3.3f Loss = %3.3f"%(y[0,0], loss))

y = 20.000 Loss = 327.371


Now let's compute the derivatives of the network.  We already computed the forward pass.  Let's compute the backward pass.

In [8]:
# We'll need the indicator function
def indicator_function(x):
  x_in = np.array(x)
  x_in[x_in>0] = 1
  x_in[x_in<=0] = 0
  return x_in

# Main backward pass routine
def backward_pass(all_weights, all_biases, all_f, all_h, y):
  # We'll store the derivatives dl_dweights and dl_dbiases in lists as well
  all_dl_dweights = [None] * (K+1)
  all_dl_dbiases = [None] * (K+1)
  # And we'll store the derivatives of the loss with respect to the activation and preactivations in lists
  all_dl_df = [None] * (K+1)
  all_dl_dh = [None] * (K+1)
  # Again for convenience we'll stick with the convention that all_h[0] is the net input and all_f[k] in the net output

  # Compute derivatives of the loss with respect to the network output
  all_dl_df[K] = np.array(d_loss_d_output(all_f[K],y))

  # Now work backwards through the network
  for layer in range(K,-1,-1):
    # TODO Calculate the derivatives of the loss with respect to the biases at layer from all_dl_df[layer]. (eq 7.22)
    # NOTE!  To take a copy of matrix X, use Z=np.array(X)
    # REPLACE THIS LINE
    all_dl_dbiases[layer] = np.array(all_dl_df[layer])

    # TODO Calculate the derivatives of the loss with respect to the weights at layer from all_dl_df[layer] and all_h[layer] (eq 7.23)
    # Don't forget to use np.matmul
    # REPLACE THIS LINE
    all_dl_dweights[layer] = np.matmul(all_dl_df[layer], all_h[layer].T)

    # TODO: calculate the derivatives of the loss with respect to the activations from weight and derivatives of next preactivations (second part of last line of eq 7.25)
    # REPLACE THIS LINE
    all_dl_dh[layer] = np.matmul(all_weights[layer].T, all_dl_df[layer])


    if layer > 0:
      # TODO Calculate the derivatives of the loss with respect to the pre-activation f (use derivative of ReLu function, first part of last line of eq. 7.25)
      # REPLACE THIS LINE
      all_dl_df[layer-1] = indicator_function(all_f[layer-1]) * all_dl_dh[layer]

  return all_dl_dweights, all_dl_dbiases

In [9]:
all_dl_dweights, all_dl_dbiases = backward_pass(all_weights, all_biases, all_f, all_h, y)

In [10]:
np.set_printoptions(precision=3)
# Make space for derivatives computed by finite differences
all_dl_dweights_fd = [None] * (K+1)
all_dl_dbiases_fd = [None] * (K+1)

# Let's test if we have the derivatives right using finite differences
delta_fd = 0.000001

# Test the dervatives of the bias vectors
for layer in range(K+1):
  dl_dbias  = np.zeros_like(all_dl_dbiases[layer])
  # For every element in the bias
  for row in range(all_biases[layer].shape[0]):
    # Take copy of biases  We'll change one element each time
    all_biases_copy = [np.array(x) for x in all_biases]
    all_biases_copy[layer][row] += delta_fd
    network_output_1, *_ = compute_network_output(net_input, all_weights, all_biases_copy)
    network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
    dl_dbias[row] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dbiases_fd[layer] = np.array(dl_dbias)
  print("-----------------------------------------------")
  print("Bias %d, derivatives from backprop:"%(layer))
  print(all_dl_dbiases[layer])
  print("Bias %d, derivatives from finite differences"%(layer))
  print(all_dl_dbiases_fd[layer])
  if np.allclose(all_dl_dbiases_fd[layer],all_dl_dbiases[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")



# Test the derivatives of the weights matrices
for layer in range(K+1):
  dl_dweight  = np.zeros_like(all_dl_dweights[layer])
  # For every element in the bias
  for row in range(all_weights[layer].shape[0]):
    for col in range(all_weights[layer].shape[1]):
      # Take copy of biases  We'll change one element each time
      all_weights_copy = [np.array(x) for x in all_weights]
      all_weights_copy[layer][row][col] += delta_fd
      network_output_1, *_ = compute_network_output(net_input, all_weights_copy, all_biases)
      network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
      dl_dweight[row][col] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dweights_fd[layer] = np.array(dl_dweight)
  print("-----------------------------------------------")
  print("Weight %d, derivatives from backprop:"%(layer))
  print(all_dl_dweights[layer])
  print("Weight %d, derivatives from finite differences"%(layer))
  print(all_dl_dweights_fd[layer])
  if np.allclose(all_dl_dweights_fd[layer],all_dl_dweights[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")

-----------------------------------------------
Bias 0, derivatives from backprop:
[[ -4.486]
 [  4.947]
 [  6.812]
 [ -3.883]
 [-24.935]
 [  0.   ]]
Bias 0, derivatives from finite differences
[[ -4.486]
 [  4.947]
 [  6.812]
 [ -3.883]
 [-24.935]
 [  0.   ]]
Success!  Derivatives match.
-----------------------------------------------
Bias 1, derivatives from backprop:
[[ -0.   ]
 [-11.297]
 [  0.   ]
 [  0.   ]
 [-10.722]
 [  0.   ]]
Bias 1, derivatives from finite differences
[[  0.   ]
 [-11.297]
 [  0.   ]
 [  0.   ]
 [-10.722]
 [  0.   ]]
Success!  Derivatives match.
-----------------------------------------------
Bias 2, derivatives from backprop:
[[-0.   ]
 [-0.   ]
 [ 0.938]
 [ 0.   ]
 [-9.993]
 [ 0.508]]
Bias 2, derivatives from finite differences
[[ 0.   ]
 [ 0.   ]
 [ 0.938]
 [ 0.   ]
 [-9.993]
 [ 0.508]]
Success!  Derivatives match.
-----------------------------------------------
Bias 3, derivatives from backprop:
[[-0.   ]
 [-4.8  ]
 [-1.661]
 [-0.   ]
 [ 3.393]
 [ 5.391]

### Análisis observado

En este notebook aprendí a llevar el algoritmo de backpropagation desde un ejemplo simple hasta una red neuronal profunda real para comprender cómo calcular las derivadas paso a paso utilizando matrices enteras de pesos y sesgos. Al desarrollar el código me di cuenta de que este proceso mantiene la misma estructura dividida en pasos principales. En el primer paso realicé un pase hacia adelante donde descompuse las operaciones capa por capa multiplicando las entradas por los pesos y sumando los sesgos antes de pasarlas por la función de activación. En el segundo paso hice el pase hacia atrás donde utilicé esos mismos resultados para retroceder desde el final de la red. Aquí comprendí que para desarmar el camino debía usar la derivada exacta de la función de activación que actúa como un simple indicador matemático para saber si dejar pasar o bloquear la señal de error. Una vez atravesada la activación procedí a empujar esta señal a través de las matrices de pesos para calcular finalmente las derivadas exactas de cada capa. Así pude confirmar que este cálculo en reversa es extremadamente rápido y necesario para obtener medidas precisas sobre el error final y que luego debemos entregarle esta información exacta a una herramienta matemática de optimización diferente para ajustar los números originales y hacer que la red aprenda verdaderamente.


---
# Sección de evaluación: retropropagación en la red de la figura 1

En la primera parte de este cuaderno usamos una red genérica con números aleatorios y activación ReLU. Ahora vamos a resolver la pregunta de evaluación adaptando este mismo proceso a la red específica de la figura 1 del enunciado, que tiene 2 entradas, capas ocultas con sigmoide y pesos fijos.

## Parámetros de la red y datos de entrenamiento

Para el paso hacia adelante, la ecuación 7.24 del libro UDL describe cómo se propagan los datos a través de la red:
$$f_0 = \boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}_i$$
$$\mathbf{h}_k = \sigma(\mathbf{f}_{k-1}), \quad k \in \{1, 2\}$$
$$\mathbf{f}_k = \boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k, \quad k \in \{1, 2\}$$

Identificamos cada componente según el diagrama:
la entrada $\mathbf{x}_i \in \mathbb{R}^2$ corresponde a los dos nodos negros;
los sesgos $\boldsymbol{\beta}_0 \in \mathbb{R}^3$, $\boldsymbol{\beta}_1 \in \mathbb{R}^2$ y $\beta_2 \in \mathbb{R}$ son los nodos verdes;
las matrices de pesos $\boldsymbol{\Omega}_0 \in \mathbb{R}^{3 \times 2}$, $\boldsymbol{\Omega}_1 \in \mathbb{R}^{2 \times 3}$ y $\boldsymbol{\Omega}_2 \in \mathbb{R}^{1 \times 2}$ están representadas por las flechas entre capas;
las preactivaciones $\mathbf{f}_0 \in \mathbb{R}^3$ y $\mathbf{f}_1 \in \mathbb{R}^2$ son las sumas ponderadas antes de la sigmoide;
las activaciones $\mathbf{h}_1 \in \mathbb{R}^3$ y $\mathbf{h}_2 \in \mathbb{R}^2$ son las salidas de las capas ocultas tras evaluar la sigmoide, mostradas en color naranja;
y $f_2 \in \mathbb{R}$ es la salida final escalar calculada en el nodo rojo.
También cargamos los 3 datos del lote de entrenamiento: $(1, 1, 1)$, $(0, 2, 1)$ y $(-1, 1, 0)$.

In [11]:
# capa 0: 2 entradas a 3 neuronas
beta_0 = np.array([[0.3],
                   [0.7],
                   [0.1]])

Omega_0 = np.array([[ 0.2, -0.1],
                    [ 0.5,  0.6],
                    [-1.3,  1.2]])

# capa 1: 3 neuronas a 2 neuronas
beta_1 = np.array([[-0.3],
                   [ 0.3]])

Omega_1 = np.array([[ 1.3,  0.6, -0.7],
                    [-0.2,  0.1,  0.4]])

# capa 2: 2 neuronas a 1 salida
beta_2 = np.array([[0.2]])

Omega_2 = np.array([[-0.9, 2.1]])

eval_weights = [Omega_0, Omega_1, Omega_2]
eval_biases = [beta_0, beta_1, beta_2]

# lote de entrenamiento con tres datos
X_batch = [
    np.array([[1.0], [1.0]]),
    np.array([[0.0], [2.0]]),
    np.array([[-1.0], [1.0]])
]

Y_batch = [
    np.array([[1.0]]),
    np.array([[1.0]]),
    np.array([[0.0]])
]

print('Parámetros y lote de entrenamiento cargados correctamente.')

Parámetros y lote de entrenamiento cargados correctamente.


Ahora definimos las funciones matemáticas que necesita la red. A diferencia de las celdas anteriores donde usamos ReLU, la figura 1 pide que las neuronas ocultas usen la función sigmoide $\sigma(t) = 1 / (1 + e^{-t})$, cuya derivada es $\sigma(t)(1 - \sigma(t))$. Para medir el error usamos la pérdida por mínimos cuadrados junto con su derivada respecto a la salida.

In [12]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def d_sigmoid(x):
    s = sigmoid(x)
    return s * (1.0 - s)

def eval_least_squares_loss(net_output, y):
    diff = net_output - y
    return np.sum(diff * diff)

def eval_d_loss_d_output(net_output, y):
    return 2.0 * (net_output - y)

## Propagación hacia adelante y cálculo de gradientes

Con las funciones listas, hacemos el paso hacia adelante procesando los 3 datos del lote uno por uno. Multiplicamos las entradas por los pesos, sumamos los sesgos y aplicamos la sigmoide en las capas intermedias, calculando al final el error de cada dato y el promedio inicial del lote.

In [13]:
def eval_forward_pass(net_input, weights, biases):
    all_f = [None] * 3
    all_h = [None] * 3
    
    all_h[0] = net_input
    
    # capa 0 con sigmoide
    all_f[0] = biases[0] + np.matmul(weights[0], all_h[0])
    all_h[1] = sigmoid(all_f[0])
    
    # capa 1 con sigmoide
    all_f[1] = biases[1] + np.matmul(weights[1], all_h[1])
    all_h[2] = sigmoid(all_f[1])
    
    # capa 2 de salida lineal
    all_f[2] = biases[2] + np.matmul(weights[2], all_h[2])
    
    return all_f[2], all_f, all_h

print('=== Propagación hacia adelante inicial ===')
total_loss = 0.0
for i in range(3):
    out, all_f, all_h = eval_forward_pass(X_batch[i], eval_weights, eval_biases)
    loss = eval_least_squares_loss(out, Y_batch[i])
    total_loss += loss
    print(f'\nDato {i+1}:')
    print(f'  x = {X_batch[i].ravel()}, y = {Y_batch[i].ravel()[0]}')
    print(f'  f_0 = {all_f[0].ravel()}')
    print(f'  h_1 = {all_h[1].ravel()}')
    print(f'  f_1 = {all_f[1].ravel()}')
    print(f'  h_2 = {all_h[2].ravel()}')
    print(f'  f_2 (salida) = {out.ravel()[0]:.6f}')
    print(f'  pérdida individual = {loss:.6f}')

loss_batch_inicial = total_loss / 3.0
print(f'\nPérdida promedio inicial del lote: {loss_batch_inicial:.6f}')

=== Propagación hacia adelante inicial ===

Dato 1:
  x = [1. 1.], y = 1.0
  f_0 = [ 4.000e-01  1.800e+00 -8.327e-17]
  h_1 = [0.599 0.858 0.5  ]
  f_1 = [0.643 0.466]
  h_2 = [0.655 0.614]
  f_2 (salida) = 0.900430
  pérdida individual = 0.009914

Dato 2:
  x = [0. 2.], y = 1.0
  f_0 = [0.1 1.9 2.5]
  h_1 = [0.525 0.87  0.924]
  f_1 = [0.258 0.652]
  h_2 = [0.564 0.657]
  f_2 (salida) = 1.072881
  pérdida individual = 0.005312

Dato 3:
  x = [-1.  1.], y = 0.0
  f_0 = [-5.551e-17  8.000e-01  2.600e+00]
  h_1 = [0.5   0.69  0.931]
  f_1 = [0.112 0.641]
  h_2 = [0.528 0.655]
  f_2 (salida) = 1.100360
  pérdida individual = 1.210792

Pérdida promedio inicial del lote: 0.408673


Para el paso hacia atrás, el libro formula en la ecuación 7.25 la regla de la cadena para la activación ReLU. Como aquí las neuronas ocultas usan la sigmoide $\sigma(t) = 1 / (1 + e^{-t})$, su derivada analítica es $\sigma'(t) = \sigma(t)(1 - \sigma(t))$. Al reemplazarla, la ecuación 7.25 modificada queda de la siguiente forma:

$$\frac{\partial \ell_i}{\partial \boldsymbol{\beta}_k} = \frac{\partial \ell_i}{\partial \mathbf{f}_k}, \quad k \in \{2, 1, 0\}$$
$$\frac{\partial \ell_i}{\partial \boldsymbol{\Omega}_k} = \frac{\partial \ell_i}{\partial \mathbf{f}_k} \mathbf{h}_k^T, \quad k \in \{2, 1, 0\} \quad (\text{con } \mathbf{h}_0 = \mathbf{x}_i)$$
$$\frac{\partial \ell_i}{\partial \mathbf{f}_{k-1}} = [\mathbf{h}_k \odot (1 - \mathbf{h}_k)] \odot \left(\boldsymbol{\Omega}_k^T \frac{\partial \ell_i}{\partial \mathbf{f}_k}\right), \quad k \in \{2, 1\}$$

donde cada componente cumple un papel específico:
$\partial \ell_i / \partial f_2 = 2(f_2 - y_i)$ es la derivada escalar del error cuadrático respecto a la salida;
$\partial \ell_i / \partial \boldsymbol{\beta}_k$ son los gradientes respecto a los sesgos de cada capa;
$\partial \ell_i / \partial \boldsymbol{\Omega}_k$ son las matrices de gradientes respecto a los pesos, obtenidas por el producto exterior entre la derivada de la preactivación y las activaciones previas traspuestas;
$\partial \ell_i / \partial \mathbf{f}_{k-1}$ es el gradiente respecto a las preactivaciones de la capa anterior;
$\boldsymbol{\Omega}_k^T (\partial \ell_i / \partial \mathbf{f}_k)$ proyecta el gradiente hacia atrás usando los pesos traspuestos;
$\mathbf{h}_k \odot (1 - \mathbf{h}_k)$ es la derivada local de la sigmoide en cada neurona oculta;
y $\odot$ representa la multiplicación elemento a elemento (producto de Hadamard).

Con estas ecuaciones calculamos los gradientes individuales para cada uno de los tres datos y luego obtenemos el gradiente promedio del lote.

In [14]:
def eval_backward_pass(weights, biases, all_f, all_h, y):
    dl_dweights = [None] * 3
    dl_dbiases = [None] * 3
    dl_df = [None] * 3
    dl_dh = [None] * 3
    
    # capa de salida
    dl_df[2] = eval_d_loss_d_output(all_f[2], y)
    dl_dbiases[2] = np.array(dl_df[2])
    dl_dweights[2] = np.matmul(dl_df[2], all_h[2].T)
    
    # capa 1 usando derivada de la sigmoide
    dl_dh[2] = np.matmul(weights[2].T, dl_df[2])
    dl_df[1] = all_h[2] * (1.0 - all_h[2]) * dl_dh[2]
    dl_dbiases[1] = np.array(dl_df[1])
    dl_dweights[1] = np.matmul(dl_df[1], all_h[1].T)
    
    # capa 0 usando derivada de la sigmoide
    dl_dh[1] = np.matmul(weights[1].T, dl_df[1])
    dl_df[0] = all_h[1] * (1.0 - all_h[1]) * dl_dh[1]
    dl_dbiases[0] = np.array(dl_df[0])
    dl_dweights[0] = np.matmul(dl_df[0], all_h[0].T)
    
    return dl_dweights, dl_dbiases

batch_dl_dw = [np.zeros_like(w) for w in eval_weights]
batch_dl_db = [np.zeros_like(b) for b in eval_biases]

for i in range(3):
    out, all_f, all_h = eval_forward_pass(X_batch[i], eval_weights, eval_biases)
    dw_i, db_i = eval_backward_pass(eval_weights, eval_biases, all_f, all_h, Y_batch[i])
    for l in range(3):
        batch_dl_dw[l] += dw_i[l] / 3.0
        batch_dl_db[l] += db_i[l] / 3.0

print('=== Gradientes promedio del lote ===')
for l in range(3):
    print(f'\nCapa {l}:')
    print(f'  dL/dbeta_{l} =\n{batch_dl_db[l]}')
    print(f'  dL/dOmega_{l} =\n{batch_dl_dw[l]}')

=== Gradientes promedio del lote ===

Capa 0:
  dL/dbeta_0 =
[[-0.07 ]
 [-0.014]
 [ 0.012]]
  dL/dOmega_0 =
[[ 0.077 -0.074]
 [ 0.014 -0.014]
 [-0.022  0.013]]

Capa 1:
  dL/dbeta_1 =
[[-0.162]
 [ 0.338]]
  dL/dOmega_1 =
[[-0.08  -0.111 -0.156]
 [ 0.166  0.232  0.329]]

Capa 2:
  dL/dbeta_2 =
[[0.716]]
  dL/dOmega_2 =
[[0.371 0.472]]


## Verificación de derivadas y descenso de gradiente

Al cambiar a mano las fórmulas analíticas de backpropagation para usar la sigmoide, es muy fácil equivocarse en un signo o en una transpuesta. Para estar completamente seguros de que nuestras derivadas son correctas, usamos el método de diferencias finitas. Le sumamos un valor muy chiquito ($\delta = 10^{-6}$) a cada peso y sesgo por separado y calculamos la derivada numérica. Si coincide con nuestro cálculo analítico, confirmamos que el código no tiene ningún error.

In [15]:
delta_fd = 1e-6

def compute_batch_loss(weights, biases):
    total = 0.0
    for i in range(3):
        out, *_ = eval_forward_pass(X_batch[i], weights, biases)
        total += eval_least_squares_loss(out, Y_batch[i])
    return total / 3.0

print('=== Comprobación con diferencias finitas ===')

# comprobación de sesgos
for l in range(3):
    fd_b = np.zeros_like(eval_biases[l])
    for r in range(eval_biases[l].shape[0]):
        b_copy = [np.array(x) for x in eval_biases]
        b_copy[l][r, 0] += delta_fd
        l1 = compute_batch_loss(eval_weights, b_copy)
        l0 = compute_batch_loss(eval_weights, eval_biases)
        fd_b[r, 0] = (l1 - l0) / delta_fd
        
    print('-----------------------------------------------')
    print(f'Sesgo {l}, derivada analítica:')
    print(batch_dl_db[l])
    print(f'Sesgo {l}, derivada numérica (diferencias finitas):')
    print(fd_b)
    if np.allclose(batch_dl_db[l], fd_b, rtol=1e-4, atol=1e-6):
        print('Resultado: coincidencia exitosa.')
    else:
        print('Resultado: discrepancia encontrada.')

# comprobación de pesos
for l in range(3):
    fd_w = np.zeros_like(eval_weights[l])
    for r in range(eval_weights[l].shape[0]):
        for c in range(eval_weights[l].shape[1]):
            w_copy = [np.array(x) for x in eval_weights]
            w_copy[l][r, c] += delta_fd
            l1 = compute_batch_loss(w_copy, eval_biases)
            l0 = compute_batch_loss(eval_weights, eval_biases)
            fd_w[r, c] = (l1 - l0) / delta_fd
            
    print('-----------------------------------------------')
    print(f'Peso {l}, derivada analítica:')
    print(batch_dl_dw[l])
    print(f'Peso {l}, derivada numérica (diferencias finitas):')
    print(fd_w)
    if np.allclose(batch_dl_dw[l], fd_w, rtol=1e-4, atol=1e-6):
        print('Resultado: coincidencia exitosa.')
    else:
        print('Resultado: discrepancia encontrada.')

=== Comprobación con diferencias finitas ===
-----------------------------------------------
Sesgo 0, derivada analítica:
[[-0.07 ]
 [-0.014]
 [ 0.012]]
Sesgo 0, derivada numérica (diferencias finitas):
[[-0.07 ]
 [-0.014]
 [ 0.012]]
Resultado: coincidencia exitosa.
-----------------------------------------------
Sesgo 1, derivada analítica:
[[-0.162]
 [ 0.338]]
Sesgo 1, derivada numérica (diferencias finitas):
[[-0.162]
 [ 0.338]]
Resultado: coincidencia exitosa.
-----------------------------------------------
Sesgo 2, derivada analítica:
[[0.716]]
Sesgo 2, derivada numérica (diferencias finitas):
[[0.716]]
Resultado: coincidencia exitosa.
-----------------------------------------------
Peso 0, derivada analítica:
[[ 0.077 -0.074]
 [ 0.014 -0.014]
 [-0.022  0.013]]
Peso 0, derivada numérica (diferencias finitas):
[[ 0.077 -0.074]
 [ 0.014 -0.014]
 [-0.022  0.013]]
Resultado: coincidencia exitosa.
-----------------------------------------------
Peso 1, derivada analítica:
[[-0.08  -0.1

Como las derivadas analíticas coincidieron de forma exacta con las diferencias finitas, aplicamos el paso de descenso de gradiente para actualizar los pesos. El enunciado nos pide usar una tasa de aprendizaje de $\alpha = 1.0$.

Al hacer la actualización, vemos que la pérdida sube de 0.4087 a 2.1498 en vez de bajar. Esto pasa porque 1.0 es un paso demasiado grande para los números de esta red y hace que la salida se pase de largo del mínimo (lo que se conoce como sobrepaso u *overshooting*). Para demostrar que el gradiente sí apunta en la dirección correcta, probamos también con una tasa más prudente ($\alpha = 0.2$), confirmando que con un paso adecuado la pérdida sí baja de forma constante a 0.2798.

In [16]:
alpha = 1.0
new_weights = [w - alpha * gw for w, gw in zip(eval_weights, batch_dl_dw)]
new_biases = [b - alpha * gb for b, gb in zip(eval_biases, batch_dl_db)]

print('=== Nuevos parámetros tras el paso (tasa = 1.0) ===')
for l in range(3):
    print(f'\nCapa {l}:')
    print(f'  beta_{l}^(nuevo) =\n{new_biases[l]}')
    print(f'  Omega_{l}^(nuevo) =\n{new_weights[l]}')

loss_initial = compute_batch_loss(eval_weights, eval_biases)
loss_after_1 = compute_batch_loss(new_weights, new_biases)
print(f'\nPérdida inicial: {loss_initial:.6f}')
print(f'Pérdida con tasa 1.0: {loss_after_1:.6f}\n')

# prueba con tasa moderada de 0.2
alpha_small = 0.2
new_w_small = [w - alpha_small * gw for w, gw in zip(eval_weights, batch_dl_dw)]
new_b_small = [b - alpha_small * gb for b, gb in zip(eval_biases, batch_dl_db)]
loss_small = compute_batch_loss(new_w_small, new_b_small)
print(f'Pérdida con tasa moderada (0.2): {loss_small:.6f}')

=== Nuevos parámetros tras el paso (tasa = 1.0) ===

Capa 0:
  beta_0^(nuevo) =
[[0.37 ]
 [0.714]
 [0.088]]
  Omega_0^(nuevo) =
[[ 0.123 -0.026]
 [ 0.486  0.614]
 [-1.278  1.187]]

Capa 1:
  beta_1^(nuevo) =
[[-0.138]
 [-0.038]]
  Omega_1^(nuevo) =
[[ 1.38   0.711 -0.544]
 [-0.366 -0.132  0.071]]

Capa 2:
  beta_2^(nuevo) =
[[-0.516]]
  Omega_2^(nuevo) =
[[-1.271  1.628]]

Pérdida inicial: 0.408673
Pérdida con tasa 1.0: 2.149793

Pérdida con tasa moderada (0.2): 0.279846


Al comparar todos los resultados de este código con los cálculos que hicimos a mano en el informe, vemos que coinciden exactamente número por número. Las señales intermedias, las activaciones de la sigmoide y las salidas de los tres puntos dieron los mismos valores. Además, la prueba de diferencias finitas confirmó que adaptar la regla de la cadena a la sigmoide calcula los gradientes de forma exacta, y pudimos comprobar en la práctica por qué una tasa de 1.0 hace que la red se pase del mínimo mientras que una tasa de 0.2 reduce la pérdida sin problemas.